In [ ]:
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from utils.graph_builder import LocalizationGraphBuilder, traverse_namespaces
from adaptation.misc import NameAnonymizer
import os


In [2]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} bert={} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/spanish_word2vec/word2vec.bin'} spacy={'es': 'es_core_news_sm'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/lstm_mean_cosine_noaug'} allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000


In [3]:
adaptation_dir = "./adaptation"

localization_dir = os.path.join(adaptation_dir, "localization")
language_dir = os.path.join(localization_dir, "final")
structure_dir = os.path.join(localization_dir,  "structure")

database_dir = "./faiss_data"

data_dir = os.path.join(adaptation_dir, "data")
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [4]:
namespaces = traverse_namespaces(language_dir, languages)
print(namespaces)


['scene1/scene1Bedroom2', 'scene6/scene6Bedroom', 'scene3/scene3Break', 'scene1/scene1Classroom', 'scene5/scene5Bedroom', 'scene1/scene1Lunch2', 'scene6/routeB/scene6PoliceStationRouteB', 'scene6/scene6Livingroom', 'scene4/scene4Bedroom', 'scene6/routeA/scene6BedroomRouteA2', 'names', 'scene4/scene4Garage', 'computer/usernames', 'scene4/scene4Frontyard', 'menus/loginScene', 'scene1/scene1Lunch1', 'deviceInfo', 'scene2/scene2Bedroom', 'scene6/routeA/scene6PortalRouteA', 'scene6/routeB/scene6LunchRouteB', 'scene1/scene1Break', 'menus/creditsScene', 'scene6/routeB/scene6BedroomRouteB', 'computer/captions', 'computer/loginScreen', 'scene4/scene4Backyard', 'dialogManager', 'computer/socialMediaScreen', 'scene7/scene7Bedroom', 'scene6/routeB/scene6EndingRouteB', 'menus/titleScene', 'transitions', 'scene3/scene3Bedroom', 'generalDialogs', 'scene5/scene5Livingroom', 'scene6/routeA/scene6EndingRouteA', 'scene6/routeA/scene6LunchRouteA', 'scene1/scene1Bedroom1', 'scene2/scene2Break', 'scene6/rou

In [5]:
backend = Backend(
    name_mapping=lambda lng, ns: os.path.join(
        language_dir,
        lng,
        f"{ns}.json"
    )
)

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [6]:
# from pyi18next.utility import get_plural_func

# # rules = "one: n is 1; other:"
# rules = {
# 	"one": "n is 1",
# 	"other": ""
# }

# plural_func = get_plural_func(rules)

# print(plural_func(1))
# print(plural_func(3))


In [7]:
name_anonymizer = NameAnonymizer(
    names_path=spanish_names_path,
    whitelist_path=name_whitelist_path,
    replacement="[UNK]"
)


In [8]:
model_registry = ModelRegistry(languages)
model_registry.build_transformer("sbert")
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
model_types = model_registry.active_model_types()


2026-06-28 04:34:54.415 | DEBUG    | services.model_registry:_create_loader:59 - Registering sbert loader for 'es'
2026-06-28 04:34:54.415 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-06-28 04:34:55.445 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'


In [9]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir=structure_dir,
)

builder.run()


2026-06-28 04:34:55.760 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 48 vectors
2026-06-28 04:34:55.760 | DEBUG    | services.node_engine:save_node:44 - Saving FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2


Total visited nodes: 665


In [10]:
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
test_engine = multilingual.get_node_engine("es", "sbert")

print(test_engine.retrievers)

test_engine.load_all()

print(test_engine.retrievers)


2026-06-28 04:34:55.780 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2
2026-06-28 04:34:55.780 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.


{}
{'scene1Classroom_part2_thanks2': <controllers.retrievers.faiss.FaissRetriever object at 0x000001C465292750>}


In [11]:
retriever = test_engine.get_retriever("scene1Classroom_part2_thanks2")

retriever.search("Hola", 3)


(array([42, 35, 17], dtype=int32),
 array([0.99999994, 0.5333565 , 0.5254723 ], dtype=float32),
 array(['Hola', 'Buenas! Soy [UNK] encantado.', 'Holaaa, soy [UNK] que ta'],
       dtype=object))